# 06 - Failure Investigator
Seed-0 test set of Titanic (179 passengers). Where do models agree, where do they fail, and what do they rely on?

In [ ]:
import sys, json; sys.path.insert(0, '../src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from arena.config import RAW_DIR, SUMMARY_DIR, PRED_DIR, FIG_DIR
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
cases = pd.read_csv(SUMMARY_DIR/'failures_clf_cases.csv')
cm = pd.read_csv(SUMMARY_DIR/'failures_clf_confusion.csv')
sub = pd.read_csv(SUMMARY_DIR/'failures_clf_subgroups.csv')
imp = pd.read_csv(SUMMARY_DIR/'failures_importance_clf.csv')
models = [c[:-5] for c in cases.columns if c.endswith('_pred')]
print(models)

## 1. Agreement structure
`hard_core` = every model is wrong; `contested` = models split; `all_correct`.

In [ ]:
cases.category.value_counts().to_frame('passengers').assign(share=lambda d: (d.passengers/d.passengers.sum()).round(3))

### Hard-core failures (all models wrong), with raw features

In [ ]:
cols = ['row_id','y_true','Pclass','Sex','Age','Fare','Embarked','Title','FamilySize','n_wrong']
cases[cases.category=='hard_core'][cols]

### Most contested passengers (models split about evenly)

In [ ]:
c = cases[cases.category=='contested'].assign(split=lambda d: (d.n_wrong - len(models)/2).abs()).sort_values('split').head(10)
c[cols]

## 2. Confusion matrices per model (rows = true, columns = predicted; 1 = survived)

In [ ]:
n = len(models); fig, axes = plt.subplots(2, (n+1)//2, figsize=(3*((n+1)//2), 5.4)); axes = axes.ravel()
for ax, m in zip(axes, models):
    t = cm[cm.model==m].pivot(index='true', columns='pred', values='count'); sns.heatmap(t, annot=True, fmt='d', cbar=False, cmap='Blues', ax=ax); ax.set_title(m)
for ax in axes[n:]: ax.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR/'confusion_matrices.png', dpi=130); plt.show()

## 3. Error rate by subgroup

In [ ]:
sub.pivot_table(index=['group','value'], columns='model', values='error_rate').round(2).join(sub.drop_duplicates(['group','value']).set_index(['group','value']).n)

## 4. What do models rely on? Permutation importance (test set, drop in macro-F1) vs. native importance

In [ ]:
p = imp.pivot(index='feature', columns='model', values='perm_importance').round(3); display(p.sort_values('xgb', ascending=False))
nat = imp.pivot(index='feature', columns='model', values='native_importance').dropna(axis=1, how='all').round(3); display(nat)

In [ ]:
print('Spearman rank agreement of permutation importance between models:'); display(imp.pivot(index='feature', columns='model', values='perm_importance').corr(method='spearman').round(2))
from scipy.stats import spearmanr
for m in nat.columns:
    print(m, 'permutation vs native rank agreement: %.2f' % spearmanr(p[m], nat[m])[0])

## 5. ID3 learned tree (from-scratch implementation, discretized features)

In [ ]:
from arena.evaluate import split
from arena.preprocess import Discretizer
from arena.id3 import ID3Classifier
Xtr, Xte, ytr, yte = split('clf', 0)
d = Discretizer().fit(Xtr); m = ID3Classifier(max_depth=3).fit(d.transform(Xtr), ytr)
print(m.export_text(list(d.cols_), ['died','survived'])); print('test acc: %.3f' % (m.predict(d.transform(Xte)) == yte).mean())

## 6. Regression failures (California, seed 0)

In [ ]:
rng = pd.read_csv(SUMMARY_DIR/'failures_reg_by_range.csv')
display(rng.pivot(index='target_range', columns='model', values='mae').round(3))
display(rng.pivot(index='target_range', columns='model', values='mean_residual').round(3))

`mean_residual = actual - predicted`: positive = under-prediction. The `capped(5.0)` row shows the ceiling effect: true prices are cut at $500k so models cannot learn them.

In [ ]:
w = pd.read_csv(SUMMARY_DIR/'failures_reg_worst.csv'); w.head(10).round(2)

In [ ]:
impr = pd.read_csv(SUMMARY_DIR/'failures_importance_reg.csv'); impr.pivot(index='feature', columns='model', values='perm_importance').round(3).sort_values('xgb', ascending=False)

## 7. Trade-off: accuracy vs. robustness vs. complexity vs. cost

In [ ]:
tc = pd.read_csv(SUMMARY_DIR/'tradeoff_clf.csv').dropna(subset=['feature_robustness']); tr = pd.read_csv(SUMMARY_DIR/'tradeoff_reg.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, t, ttl in zip(axes, [tc, tr], ['Titanic (macro-F1)', 'California (RMSE, lower better)']):
    ax.scatter(t.score, t.feature_robustness, s=60 + 25*t.fit_time_s.clip(upper=20), alpha=.6)
    for r in t.itertuples(): ax.annotate(f'{r.model} (c={r.complexity})', (r.score, r.feature_robustness), fontsize=8)
    ax.set_xlabel('clean score'); ax.set_ylabel('mean rel. change under noise/missing/drop (higher = more robust)'); ax.set_title(ttl)
plt.tight_layout(); plt.savefig(FIG_DIR/'tradeoff.png', dpi=130); plt.show()
display(tc.round(3)); display(tr.round(3))

In [ ]:
# Data-driven summary (no hand-written numbers)
best = tc.sort_values('score').iloc[-1]; most_rob = tc.sort_values('feature_robustness').iloc[-1]
print(f"Titanic: best clean model = {best.model} ({best.score:.3f}); most feature-robust = {most_rob.model} ({most_rob.feature_robustness:+.3f}, clean score {most_rob.score:.3f}).")
br = tr.sort_values('score').iloc[0]; mr = tr.sort_values('feature_robustness').iloc[-1]
print(f"California: best clean model = {br.model} (RMSE {br.score:.3f}); most feature-robust = {mr.model} ({mr.feature_robustness:+.2f}, RMSE {mr.score:.3f}).")